# E01 Matrix Sensing Benchmark — PyTorch

This notebook is the PyTorch notebook-first rewrite of the Matrix Sensing benchmark. It keeps the experiment visible in one place:

- target matrix generation
- measurement tensor generation
- matrix sensing loss
- optimizer selection
- run loop
- metrics
- plots
- conclusion

Official `torch.optim.Muon` is used when available; local exact-SVD Muon and Shampoo helpers come from `muonlib_torch`.

Default mode is the full E01 grid: 5 optimizers, 3 dimensions, 10 seeds, and 2000 iterations per run. Set `SMOKE_MODE = True` in the parameter cell only for a quick sanity check. Results stay in notebook memory as `df` and `trajectories`; this notebook does not write CSV, PNG, or report files.

In [ ]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Markdown, clear_output, display

PROJECT = Path.cwd().resolve()
if not (PROJECT / "muonlib_torch").exists():
    PROJECT = PROJECT.parent.resolve()
sys.path.insert(0, str(PROJECT))

from muonlib_torch import MuonTorch, ShampooTorch

torch.set_default_dtype(torch.float64)

print(f"project = {PROJECT}")
print(f"torch   = {torch.__version__}")

## Parameters

These constants define the experiment. The notebook should make the run grid obvious without opening any library code.

In [ ]:
# Set this to True only for a quick local sanity check.
SMOKE_MODE = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float64

ALGOS = ["Muon", "Muon-Exact", "Shampoo", "Adam", "SGD"]
DIMS = [30] if SMOKE_MODE else [50, 60, 70]
RANK = 5
LR = 0.01
NOISE = 0.0
DIST = "normal"
SPECTRUM = "hard-cutoff"
KAPPA = 1.0
INIT_SCALE = 0.01
ITERS = 120 if SMOKE_MODE else 2000
SEEDS = list(range(2)) if SMOKE_MODE else list(range(10))
EPSILON = 0.01

print(f"device={DEVICE}, dtype={DTYPE}, smoke={SMOKE_MODE}")
print(f"runs={len(ALGOS) * len(DIMS) * len(SEEDS)}, iters={ITERS}")

In [ ]:
def grid_preview():
    rows = []
    for algo in ALGOS:
        for d in DIMS:
            m_meas = 2 * d * RANK
            rows.append({
                "algo": algo,
                "d": d,
                "rank": RANK,
                "m_meas": m_meas,
                "seeds": len(SEEDS),
                "iters_per_seed": ITERS,
                "total_steps": len(SEEDS) * ITERS,
            })
    return pd.DataFrame(rows)

display(grid_preview())

## Matrix Sensing Problem

Target matrix:

$$
X^\star = U \operatorname{diag}(s)V^\top
$$

Measurements:

$$
y_i = \langle A_i, X^\star\rangle + \varepsilon_i
$$

Loss:

$$
f(X) = \frac{1}{2m}\sum_{i=1}^{m}(\langle A_i, X\rangle-y_i)^2
$$

The key PyTorch expression is `torch.einsum("mij,ij->m", A, X)`, which computes all matrix inner products at once.

In [ ]:
def make_generator(seed):
    try:
        return torch.Generator(device=DEVICE).manual_seed(int(seed))
    except Exception:
        return torch.Generator().manual_seed(int(seed))


def randn(shape, seed):
    return torch.randn(shape, generator=make_generator(seed), device=DEVICE, dtype=DTYPE)


def generate_target_matrix(d, rank, spectrum="hard-cutoff", kappa=1.0, seed=0):
    U, _ = torch.linalg.qr(randn((d, d), seed))
    V, _ = torch.linalg.qr(randn((d, d), seed + 17))

    s = torch.zeros(d, device=DEVICE, dtype=DTYPE)
    if spectrum == "hard-cutoff":
        s[:rank] = 1.0
    elif spectrum == "polynomial-decay":
        idx = torch.arange(1, rank + 1, device=DEVICE, dtype=DTYPE)
        s[:rank] = idx.pow(-1.0)
        s[:rank] /= s[0]
    elif spectrum == "exponential-decay":
        idx = torch.arange(rank, device=DEVICE, dtype=DTYPE)
        s[:rank] = torch.exp(-0.5 * idx)
    else:
        raise ValueError(f"unknown spectrum: {spectrum}")

    if kappa > 1.0 and rank > 1:
        s[:rank] = torch.linspace(1.0, 1.0 / kappa, rank, device=DEVICE, dtype=DTYPE)

    return U @ torch.diag(s) @ V.T


def generate_measurements(d, m_meas, dist="normal", seed=0, sparsity=0.1):
    g = make_generator(seed)
    shape = (m_meas, d, d)

    if dist == "normal":
        A = torch.randn(shape, generator=g, device=DEVICE, dtype=DTYPE)
    elif dist == "uniform":
        A = 2.0 * torch.rand(shape, generator=g, device=DEVICE, dtype=DTYPE) - 1.0
    elif dist == "rademacher":
        A = torch.randint(0, 2, shape, generator=g, device=DEVICE).to(DTYPE)
        A = A.mul(2.0).sub(1.0)
    elif dist == "sparse":
        dense = torch.randn(shape, generator=g, device=DEVICE, dtype=DTYPE)
        mask = torch.rand(shape, generator=g, device=DEVICE, dtype=DTYPE) < sparsity
        A = dense * mask / (sparsity * d * d) ** 0.5
    elif dist == "sphere":
        A = torch.randn(shape, generator=g, device=DEVICE, dtype=DTYPE)
        A = A / A.flatten(1).norm(dim=1).clamp_min(1e-12).view(m_meas, 1, 1)
    else:
        raise ValueError(f"unknown measurement distribution: {dist}")
    return A


def matrix_sensing_loss(A, y, X):
    pred = torch.einsum("mij,ij->m", A, X)
    return 0.5 * torch.mean((pred - y) ** 2)

## Optimizers

`Muon` uses official `torch.optim.Muon` when available and falls back to `MuonTorch` on older PyTorch; `Muon-Exact` uses local exact-SVD `MuonTorch`; `Shampoo` uses `ShampooTorch`; `Adam` and `SGD` use built-in PyTorch optimizers. The optimizer choice is intentionally visible here.

In [ ]:
def make_optimizer(algo, params, lr):
    if algo == "Muon":
        if hasattr(torch.optim, "Muon"):
            return torch.optim.Muon(
                params,
                lr=lr,
                weight_decay=0.0,
                momentum=0.9,
                nesterov=False,
                ns_steps=5,
            )
        return MuonTorch(params, lr=lr, momentum=0.9, variant="newton_schulz", ns_steps=5)
    if algo in {"Muon-Exact", "MuonExact"}:
        return MuonTorch(params, lr=lr, momentum=0.9, variant="exact")
    if algo == "Muon-RandSVD":
        return MuonTorch(params, lr=lr, momentum=0.9, variant="randsvd", rank=RANK)
    if algo == "Muon-Trunc":
        return MuonTorch(params, lr=lr, momentum=0.9, variant="truncated", rank=RANK)
    if algo == "Shampoo":
        return ShampooTorch(params, lr=lr, beta2=0.9, epsilon=1e-8)
    if algo == "Adam":
        return torch.optim.Adam(params, lr=lr)
    if algo == "SGD":
        return torch.optim.SGD(params, lr=lr, momentum=0.9)
    raise ValueError(f"unknown algo: {algo}")


def sync_device():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

## Single Run

This mirrors the old scripts in one important way: `K_epsilon` is recorded when the loss first crosses the threshold, but the loop does not break early. That keeps all runs at the same iteration budget for plotting.

In [ ]:
def run_ms_once(algo, d, rank, lr, noise, dist, spectrum, kappa, init_scale, seed, iters):
    X_star = generate_target_matrix(d, rank, spectrum=spectrum, kappa=kappa, seed=seed)
    m_meas = int(2 * d * rank)
    A = generate_measurements(d, m_meas, dist=dist, seed=seed + 1000)
    y = torch.einsum("mij,ij->m", A, X_star)
    if noise > 0:
        y = y + randn((m_meas,), seed + 2000) * noise

    X0 = randn((d, d), seed + 3000) * init_scale
    X = torch.nn.Parameter(X0)
    opt = make_optimizer(algo, [X], lr)

    losses = []
    grad_norms = []
    k_epsilon = None

    sync_device()
    t0 = time.time()
    for step in range(iters):
        opt.zero_grad(set_to_none=True)
        loss = matrix_sensing_loss(A, y, X)
        loss.backward()

        grad_norm = float(X.grad.detach().norm().cpu())
        opt.step()

        loss_value = float(loss.detach().cpu())
        losses.append(loss_value)
        grad_norms.append(grad_norm)

        if k_epsilon is None and loss_value <= EPSILON:
            k_epsilon = step + 1

    sync_device()
    elapsed = time.time() - t0
    if k_epsilon is None:
        k_epsilon = iters + 1

    row = {
        "algo": algo,
        "d": d,
        "r": rank,
        "m_meas": m_meas,
        "lr": lr,
        "noise": noise,
        "dist": dist,
        "spectrum": spectrum,
        "kappa": kappa,
        "init_scale": init_scale,
        "seed": seed,
        "iters": iters,
        "final_loss": losses[-1],
        "min_loss": min(losses),
        "K_epsilon": k_epsilon,
        "time_s": elapsed,
    }
    trajectory = {"loss": losses, "grad_norm": grad_norms}
    return row, trajectory

## Metrics and Plots

The result table is one row per run. The trajectory dictionary stores per-step loss and gradient norm for plotting inside the notebook.

In [ ]:
def summary_table(df):
    return df.groupby(["algo", "d"], as_index=False).agg(
        runs=("seed", "count"),
        K_epsilon_mean=("K_epsilon", "mean"),
        K_epsilon_std=("K_epsilon", "std"),
        min_loss_mean=("min_loss", "mean"),
        final_loss_mean=("final_loss", "mean"),
        time_s_mean=("time_s", "mean"),
    )


def trajectory_frame(trajectories):
    records = []
    for (algo, d, seed), traj in trajectories.items():
        for step, loss in enumerate(traj["loss"]):
            records.append({"algo": algo, "d": d, "seed": seed, "step": step, "loss": loss})
    return pd.DataFrame(records)


def plot_loss_curves(trajectories):
    if not trajectories:
        return
    tf = trajectory_frame(trajectories)
    mean_tf = tf.groupby(["algo", "d", "step"], as_index=False)["loss"].mean()

    plt.figure(figsize=(8.5, 4.8))
    for (algo, d, seed), traj in list(trajectories.items())[:12]:
        plt.plot(traj["loss"], color="0.75", linewidth=1, alpha=0.45)
    for (algo, d), sub in mean_tf.groupby(["algo", "d"]):
        plt.plot(sub["step"], sub["loss"], linewidth=2.4, label=f"{algo} d={d} mean")
    plt.yscale("log")
    plt.xlabel("step")
    plt.ylabel("loss")
    plt.title("Matrix sensing loss trajectories")
    plt.legend(fontsize=8)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_summary(df):
    if df.empty:
        return
    summary = summary_table(df)
    display(df.sort_values(["algo", "d", "seed"]).reset_index(drop=True))
    display(summary)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for algo, sub in summary.groupby("algo"):
        axes[0].plot(sub["d"], sub["K_epsilon_mean"], marker="o", label=algo)
        axes[1].plot(sub["d"], sub["time_s_mean"], marker="o", label=algo)
    axes[0].set_title("Mean K_epsilon")
    axes[0].set_xlabel("d")
    axes[0].set_ylabel("iterations")
    axes[1].set_title("Mean wall-clock")
    axes[1].set_xlabel("d")
    axes[1].set_ylabel("seconds")
    for ax in axes:
        ax.grid(alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.show()

## Run the Experiment

`run_experiment(live=True)` updates the table and plots after each run. For the full E01 grid, live plotting is convenient but slower; use `live=False` for raw throughput.

In [ ]:
def run_experiment(live=False):
    rows = []
    trajectories = {}
    total = len(ALGOS) * len(DIMS) * len(SEEDS)
    idx = 0

    for algo in ALGOS:
        for d in DIMS:
            for seed in SEEDS:
                idx += 1
                print(f"[{idx}/{total}] {algo} d={d} seed={seed}", flush=True)
                row, traj = run_ms_once(
                    algo=algo,
                    d=d,
                    rank=RANK,
                    lr=LR,
                    noise=NOISE,
                    dist=DIST,
                    spectrum=SPECTRUM,
                    kappa=KAPPA,
                    init_scale=INIT_SCALE,
                    seed=seed,
                    iters=ITERS,
                )
                rows.append(row)
                trajectories[(algo, d, seed)] = traj

                if live:
                    clear_output(wait=True)
                    print(f"completed {idx}/{total}")
                    df_live = pd.DataFrame(rows)
                    plot_summary(df_live)
                    plot_loss_curves(trajectories)

    df = pd.DataFrame(rows)
    return df, trajectories

## Execute

Run the next cell to start the experiment. It leaves results in memory only:

- `df`: one row per run
- `trajectories`: per-step loss and gradient norm

In [ ]:
df, trajectories = run_experiment(live=False)
plot_summary(df)
plot_loss_curves(trajectories)

## Conclusion

This cell derives a short conclusion from the in-memory result table.

In [ ]:
def conclusion_markdown(df):
    summary = summary_table(df)
    lines = []
    lines.append("### Result Summary")
    lines.append("")
    lines.append(f"- Runs: `{len(df)}`")
    lines.append(f"- Methods: `{', '.join(sorted(df['algo'].unique()))}`")
    lines.append(f"- Iterations per run: `{int(df['iters'].iloc[0])}`")
    lines.append(f"- Total optimizer steps: `{len(df) * int(df['iters'].iloc[0])}`")
    lines.append("")

    for d in sorted(df["d"].unique()):
        sub = summary[summary["d"] == d]
        best_k = sub.loc[sub["K_epsilon_mean"].idxmin()]
        best_time = sub.loc[sub["time_s_mean"].idxmin()]
        best_loss = sub.loc[sub["min_loss_mean"].idxmin()]
        lines.append(
            f"- d={d}: best K_epsilon is `{best_k.algo}` at `{best_k.K_epsilon_mean:.1f}` steps; "
            f"fastest wall-clock is `{best_time.algo}` at `{best_time.time_s_mean:.3f}s`; "
            f"lowest min_loss is `{best_loss.algo}` at `{best_loss.min_loss_mean:.3e}`."
        )

    if {"Muon", "Muon-Exact"}.issubset(set(df["algo"])):
        lines.append("")
        lines.append("Muon vs Muon-Exact checks whether the Newton-Schulz approximation tracks the exact SVD polar direction.")
    if "Shampoo" in set(df["algo"]):
        lines.append("Shampoo is included as a second-order preconditioned baseline; compare both K_epsilon and wall-clock, because its eigendecompositions are not free.")
    return "\n".join(lines)


display(Markdown(conclusion_markdown(df)))